In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

data_dir = Path("../data")  
files = sorted(data_dir.rglob("raw_*.csv"))

series_list = []

for path in files:
    asset = path.stem.replace("raw_", "")

    df = pd.read_csv(path, usecols=["Date", "Close"])
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

    s = df.set_index("Date")["Close"]
    s = s.where(s > 0, np.nan)
    log_returns = np.log(s / s.shift(1)).rename(asset)

    series_list.append(log_returns)

merged = pd.concat(series_list, axis=1)
merged =merged.dropna()
merged = merged.reset_index()
merged = merged.sort_values("Date").reset_index(drop=True)

merged["Date"] = merged["Date"].dt.strftime("%Y-%m-%d")
merged.to_csv(data_dir / "all_assets_log_returns.csv", index=False)

merged.head()
print(merged.shape)



(3760, 31)
